# Marginals Obtaining Test Notebook
This notebook verifies the `TopKObtainer` implementation, ensuring it correctly selects marginals based on utility and adds noise.

## 1. Setup & Orchestration

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

from src.loading.file_loader import FileLoader
from src.loading.components.data_loader import DataLoader
from src.loading.components.dcs_loader import DCsLoader
from src.loading.components.metadata_loader import MetadataLoader
from src.loading.components.data_encoder import DataEncoder
from src.loading.components.dcs_encoder import DCsEncoder
from src.synthesizing.co_noise import CoNoise
from src.marginals_obtaining.top_k_obtainer import TopKObtainer
from src.marginals_obtaining.utility_functions.distance_utility import DistanceUtility

def get_dataset(dataset_name):
    loader = FileLoader(
        name=dataset_name, 
        base_path="../data",
        data_loader=DataLoader(),
        dcs_loader=DCsLoader(),
        metadata_loader=MetadataLoader(),
        data_encoder=DataEncoder(),
        dcs_encoder=DCsEncoder()
    )
    return loader.load()

## 2. Generate Synthetic Data
We need both private and synthetic data to obtain marginals.

In [ ]:
private_dataset = get_dataset("adult")
synthesizer = CoNoise(num_of_iterations=100)
synthetic_dataset = synthesizer.synthesize(private_dataset)

print(f"Private data size: {len(private_dataset.data)}")
print(f"Synthetic data size: {len(synthetic_dataset.data)}")

## 3. Obtain Marginals
Use `TopKObtainer` to get the top 20 marginals with high distance.

In [ ]:
k = 20
obtainer = TopKObtainer(
    selection_budget=1.0,
    generation_budget=1.0,
    k=k,
    utility_function=DistanceUtility()
)

marginal_set = obtainer.obtain(private_dataset, synthetic_dataset)

print(f"Obtained {len(marginal_set)} marginals.")
assert len(marginal_set) == k, f"Expected {k} marginals, got {len(marginal_set)}"

for i, m in enumerate(list(marginal_set)[:5]):
    print(f"Marginal {i}: {m.attr1}={m.value1}, {m.attr2}={m.value2} | Target: {m.target:.4f}")

## 4. Runtime Test

In [ ]:
import time

start_time = time.time()
obtainer.obtain(private_dataset, synthetic_dataset)
end_time = time.time()

print(f"Marginals obtaining runtime: {end_time - start_time:.4f}s")